# AutoValue AI — ML Training Notebook

**Weeks 1–4: ML Numeric Data Block**

This notebook trains and evaluates the price prediction model.

Pipeline:
1. Load UK Used Cars dataset
2. Exploratory Data Analysis (EDA)
3. Preprocessing + Feature Engineering
4. Model Training (Linear → RandomForest → GradientBoosting → MLP)
5. Cross-validation + Hyperparameter Tuning
6. Feature Importance
7. Save artefacts

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.ml_block import load_data, engineer_features, build_feature_matrix, evaluate_model

sns.set_theme(style='whitegrid')
DATA_DIR = '../data'
print('Libraries loaded')

## 1. Load Data

Dataset: [UK Used Cars](https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes)

Place all make-specific CSV files in `data/` before running.

In [ ]:
df = load_data(DATA_DIR)
print(f'Shape: {df.shape}')
print(f'Makes: {df["make"].unique().tolist()}')
df.head()

## 2. EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('AutoValue AI — Exploratory Data Analysis', fontsize=14)

# Price distribution
axes[0,0].hist(df['price'], bins=50, color='steelblue', edgecolor='white')
axes[0,0].set_title('Price Distribution (GBP)')
axes[0,0].set_xlabel('Price')

# Mileage vs Price
axes[0,1].scatter(df['mileage'], df['price'], alpha=0.1, s=5, color='steelblue')
axes[0,1].set_title('Mileage vs Price')
axes[0,1].set_xlabel('Mileage (km)')
axes[0,1].set_ylabel('Price (GBP)')

# Year vs Price
axes[0,2].scatter(df['year'], df['price'], alpha=0.1, s=5, color='tomato')
axes[0,2].set_title('Year vs Price')
axes[0,2].set_xlabel('Year')

# Price by fuel type
df.groupby('fuel_type')['price'].median().sort_values().plot.bar(ax=axes[1,0], color='steelblue')
axes[1,0].set_title('Median Price by Fuel Type')
axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(), rotation=30)

# Price by make (top 8)
top_makes = df.groupby('make')['price'].median().nlargest(8)
top_makes.sort_values().plot.barh(ax=axes[1,1], color='tomato')
axes[1,1].set_title('Median Price by Make (Top 8)')

# Price by transmission
df.groupby('transmission')['price'].median().plot.bar(ax=axes[1,2], color='mediumseagreen')
axes[1,2].set_title('Median Price by Transmission')
axes[1,2].set_xticklabels(axes[1,2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('../demo/eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering

In [ ]:
df_eng = engineer_features(df)
print('New features:')
print(df_eng[['year', 'mileage', 'car_age', 'km_per_year', 'is_luxury', 'is_sport', 'price']].describe())

## 4. Model Training

This runs the full `train_and_save()` pipeline from `src/ml_block.py`.

In [ ]:
from src.ml_block import train_and_save

results = train_and_save(DATA_DIR)

## 5. Model Comparison

In [ ]:
import pandas as pd
results_df = pd.DataFrame(results).T.round(2)
results_df.index.name = 'Model'
results_df = results_df.sort_values('RMSE')
print(results_df.to_string())

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results_df['RMSE'].plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('RMSE by Model (lower = better)')
axes[0].set_xlabel('RMSE (GBP)')

results_df['R2'].plot.barh(ax=axes[1], color='mediumseagreen')
axes[1].set_title('R² by Model (higher = better)')
axes[1].set_xlabel('R²')

plt.tight_layout()
plt.savefig('../demo/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Importance

In [ ]:
import joblib

model = joblib.load('../models/best_model.joblib')
feature_cols = joblib.load('../models/feature_cols.joblib')

importances = pd.Series(model.feature_importances_, index=feature_cols)
importances_sorted = importances.sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances_sorted.plot.barh(color='steelblue')
plt.title('Feature Importance (GradientBoosting)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../demo/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Test Inference

In [ ]:
from src.ml_block import predict_price

examples = [
    ('BMW', '3 Series', 2019, 45000, 'Diesel', 'Automatic', 0.85),
    ('Volkswagen', 'Golf', 2018, 60000, 'Petrol', 'Manual', 0.70),
    ('Audi', 'A4', 2020, 30000, 'Diesel', 'Automatic', 0.90),
    ('Ford', 'Focus', 2017, 80000, 'Petrol', 'Manual', 0.60),
]

for make, model_name, year, mileage, fuel, trans, cond in examples:
    price = predict_price(make, model_name, year, mileage, fuel, trans, cond)
    print(f'{make} {model_name} ({year}, {mileage:,}km, cond={cond}): GBP {price:,.0f}')